# Aula 09 — Desafios: MCP e multiagentes

**Ferramentas que seu agente descobre.**

Até agora, catálogo, frete e total moravam **dentro** do notebook. Com o MCP (Model Context Protocol),
as funções passam a morar num **servidor** e a nossa aplicação as **descobre** e chama por uma interface comum.

| Papel | Aqui no notebook |
|---|---|
| **Host** | nossa aplicação: conversa, modelo Ollama, permissões e limites |
| **Cliente MCP** | `ClientSession`: mantém a conexão e envia `tools/list` e `tools/call` |
| **Servidor MCP** | `servidor_mcp/loja.py` (FastMCP), iniciado por **stdio** dentro da sessão |

| Desafio | Tema |
|---|---|
| 1 | Publicar o servidor e descobrir o que ele oferece |
| 2 | Executar uma operação e ler o retorno (inclusive erro) |
| 3 | Ponte MCP → Ollama e o laço com permissões e limites |
| 4 | O servidor ganhou uma função: `consultar_prazo` |
| 5 | Dividir em papéis: executor e revisor |
| 6 | Papéis em paralelo |
| 7 | Sua versão |

**Antes de tudo:** rode a seção *Preparação*, com o segredo `OLLAMA_API_KEY` configurado.
Neste notebook usamos `await` direto nas células (funciona no Colab e no Jupyter/VS Code).

---
# Preparação

In [ ]:
%pip install -q "mcp==1.30.0" ollama

In [ ]:
import os

try:
    from google.colab import userdata
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except ImportError:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY')
    if not OLLAMA_API_KEY:
        from getpass import getpass
        OLLAMA_API_KEY = getpass('OLLAMA_API_KEY: ')

In [ ]:
import asyncio
import json
import sys
import time
from contextlib import asynccontextmanager
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from ollama import AsyncClient, ResponseError

HOST = 'https://ollama.com'
cliente = AsyncClient(host=HOST, headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'}, timeout=60)

MODELO_EXECUTOR = 'gpt-oss:20b'
MODELO_REVISOR = 'gpt-oss:20b'

MAX_RODADAS = 6
MAX_CHAMADAS_POR_RODADA = 4
MAX_CHAMADAS_TOTAL = 10
TIMEOUT_FERRAMENTA = 10

---
# Desafios

## Desafio 1 — Publicar o servidor e descobrir o que ele oferece

O servidor é um arquivo Python separado. O FastMCP lê **função + tipos + docstring** e monta o esquema de cada ferramenta.
A validação continua **dentro** do servidor: `quantidade` é inteiro estrito (`True` e `"2"` são recusados) e precisa estar entre 1 e 100.

Além de **tools**, um servidor pode oferecer **resources** (dados) e **prompts** (modelos de mensagem).

In [ ]:
pasta_servidor = Path('servidor_mcp')
pasta_servidor.mkdir(exist_ok=True)
arquivo_servidor = pasta_servidor / 'loja.py'

SERVIDOR_V1 = """
'''Servidor MCP da loja da turma (valores fictícios). Roda por stdio.'''
import math
from typing import Annotated, Any

from pydantic import Field
from mcp.server.fastmcp import FastMCP

mcp = FastMCP('Loja da turma')

CATALOGO = {'camiseta': 35, 'caneca': 22}
FRETES = {'centro': 8, 'alvorada': 12}


@mcp.tool()
def consultar_produto(item: str) -> dict[str, Any]:
    '''Consulta o preço unitário, em reais, de um item do catálogo fictício.
    Devolve encontrado=false e as opções quando o item não existe.'''
    nome = item.strip().lower()
    if nome not in CATALOGO:
        return {'encontrado': False, 'item': nome, 'opcoes': sorted(CATALOGO)}
    return {'encontrado': True, 'item': nome, 'preco_unitario': CATALOGO[nome]}


@mcp.tool()
def consultar_frete(bairro: str) -> dict[str, Any]:
    '''Consulta o frete, em reais, para um bairro. O frete é cobrado UMA vez por pedido.
    Devolve encontrado=false e os bairros atendidos quando o bairro não é atendido.'''
    nome = bairro.strip().lower()
    if nome not in FRETES:
        return {'encontrado': False, 'bairro': nome, 'atendidos': sorted(FRETES)}
    return {'encontrado': True, 'bairro': nome, 'frete_por_pedido': FRETES[nome]}


@mcp.tool()
def calcular_total(quantidade: Annotated[int, Field(strict=True)], preco: float, frete: float) -> float:
    '''Calcula o total do orçamento em reais: quantidade * preco + frete.
    Use depois de consultar o preço e o frete. quantidade é um inteiro de 1 a 100.'''
    if not 1 <= quantidade <= 100:
        raise ValueError('quantidade deve estar entre 1 e 100')
    for nome, valor in (('preco', preco), ('frete', frete)):
        if not math.isfinite(valor) or valor < 0:
            raise ValueError(f'{nome} deve ser um número finito e não negativo')
    return round(quantidade * preco + frete, 2)


@mcp.resource('loja://regras')
def regras() -> str:
    '''Regras de atendimento da loja.'''
    return 'Frete cobrado uma vez por pedido. Sem descontos. Valores fictícios.'


@mcp.prompt()
def orcamento(item: str, quantidade: int, bairro: str) -> str:
    '''Modelo de pedido de orçamento.'''
    return f'Quanto custam {quantidade} unidades de {item} com entrega em {bairro}?'

if __name__ == '__main__':
    mcp.run()
"""

arquivo_servidor.write_text(SERVIDOR_V1, encoding='utf-8')
print('servidor escrito em', arquivo_servidor.resolve())

O cliente **inicia o processo** do servidor e conversa por entrada/saída padrão (stdio).
O `stderr` do servidor vai para um arquivo de log — isso evita o erro `fileno` no Colab.

`conectar()` abre uma conexão nova a cada uso: o servidor é iniciado de novo e lê o arquivo atual.

In [ ]:
parametros = StdioServerParameters(command=sys.executable, args=[str(arquivo_servidor)])


@asynccontextmanager
async def conectar():
    log = pasta_servidor / 'mcp_stderr.log'
    with log.open('a', encoding='utf-8') as err:
        async with stdio_client(parametros, errlog=err) as (leitura, escrita):
            async with ClientSession(leitura, escrita) as sessao:
                info = await sessao.initialize()
                sessao.info_servidor = info
                yield sessao


async def experimentar():
    async with conectar() as sessao:
        info = sessao.info_servidor
        print(f'servidor: {info.serverInfo.name} | protocolo MCP {info.protocolVersion}\n')

        lista = await sessao.list_tools()
        for f in lista.tools:
            print(f'TOOL {f.name}: {f.description.splitlines()[0]}')
            print('     entrada:', json.dumps(f.inputSchema['properties'], ensure_ascii=False))

        for r in (await sessao.list_resources()).resources:
            print(f'\nRESOURCE {r.uri}:', (await sessao.read_resource(r.uri)).contents[0].text)
        for p in (await sessao.list_prompts()).prompts:
            print(f'PROMPT {p.name}:', [a.name for a in p.arguments])


await experimentar()

## Desafio 2 — Executar uma operação e ler o retorno

| Campo | O que traz |
|---|---|
| `content` | blocos de texto (e outros tipos) |
| `structuredContent` | dados estruturados, quando oferecidos — um `float` vira `{'result': 78.0}` |
| `isError` | `True` quando a execução da ferramenta falhou |

`ler_resultado` traduz isso para o formato que já usamos: `{'ok': ..., 'dados'/'erro': ...}`.

In [ ]:
def ler_resultado(resultado):
    texto = '\n'.join(c.text for c in resultado.content if getattr(c, 'type', '') == 'text')
    if resultado.isError:
        return {'ok': False, 'erro': texto}
    dados = resultado.structuredContent if resultado.structuredContent is not None else texto
    return {'ok': True, 'dados': dados}


async def testar_chamadas():
    async with conectar() as sessao:
        r = await sessao.call_tool('calcular_total', {'quantidade': 2, 'preco': 35, 'frete': 8})
        print('structuredContent:', r.structuredContent, '| isError:', r.isError)

        testes = [
            ('consultar_produto', {'item': 'Camiseta'}),
            ('consultar_produto', {'item': 'bermuda'}),
            ('calcular_total', {'quantidade': -2, 'preco': 35, 'frete': 8}),
            ('calcular_total', {'quantidade': True, 'preco': 35, 'frete': 8}),
            ('apagar_tudo', {}),
        ]
        for nome, args in testes:
            print(f'\n{nome}({args})\n  ->', ler_resultado(await sessao.call_tool(nome, args)))


await testar_chamadas()

**Para pensar:** o erro de quantidade negativa atravessou a conexão como `isError=True`. Quem deve decidir o próximo passo — o servidor ou a aplicação?

## Desafio 3 — Ponte MCP → Ollama e o laço com permissões e limites

O MCP fornece o esquema; a aplicação o **converte** para o formato de ferramentas do Ollama.
O modelo não abre a conexão MCP sozinho: **o laço continua sendo nosso**.

1. Envie ao modelo só as ferramentas **autorizadas** (`PERMITIDAS`).
2. Confira nome e argumentos pedidos.
3. Execute por `sessao.call_tool` (com tempo limite) e guarde o retorno.
4. Pare na resposta final, no erro ou no limite.

In [ ]:
PERMITIDAS = {'consultar_produto', 'consultar_frete', 'calcular_total'}

SYSTEM_EXECUTOR = (
    'Você é o atendente de uma loja. Responda em português, de forma curta. '
    'Use as ferramentas para preços, fretes e totais; nunca invente valores. '
    'Se um resultado vier com ok=false ou encontrado=false, explique ao cliente e não trate como sucesso. '
    'Se uma quantidade for inválida, peça uma quantidade válida.'
)


def mcp_para_ollama(lista, permitidas):
    tools = []
    for f in lista.tools:
        if f.name not in permitidas:
            continue
        tools.append({'type': 'function', 'function': {
            'name': f.name, 'description': f.description or '', 'parameters': f.inputSchema}})
    return tools


def falha(tipo, erro):
    return {'ok': False, 'tipo': tipo, 'erro': erro}


async def chamar_modelo(**kwargs):
    try:
        return await cliente.chat(**kwargs)
    except ResponseError as e:
        motivos = {401: 'confira a chave', 403: 'sem permissão para o modelo',
                   404: 'modelo não encontrado', 429: 'cota atingida; tente mais tarde'}
        raise RuntimeError(f"{e.status_code}: {motivos.get(e.status_code, e.error)}") from e


async def rodar_executor(sessao, historico, permitidas=None, verbose=True):
    permitidas = PERMITIDAS if permitidas is None else permitidas
    tools = mcp_para_ollama(await sessao.list_tools(), permitidas)
    estado = {'rodadas': 0, 'chamadas': 0, 'evidencias': []}

    def fim(status, resposta='', motivo=''):
        return {'status': status, 'resposta': resposta, 'motivo': motivo, **estado}

    for rodada in range(1, MAX_RODADAS + 1):
        estado['rodadas'] = rodada
        try:
            response = await chamar_modelo(model=MODELO_EXECUTOR, messages=historico, tools=tools,
                                           options={'num_predict': 1024})
        except Exception as e:
            return fim('erro_modelo', motivo=str(e))
        m = response.message
        chamadas = m.tool_calls or []
        historico.append({'role': 'assistant', 'content': m.content or '',
                          'tool_calls': [{'function': {'name': c.function.name,
                                                       'arguments': dict(c.function.arguments or {})}}
                                         for c in chamadas]})
        if not chamadas:
            return fim('ok', resposta=m.content or '')

        recusa = None
        if len(chamadas) > MAX_CHAMADAS_POR_RODADA:
            recusa = f'lote de {len(chamadas)} chamadas recusado (limite {MAX_CHAMADAS_POR_RODADA} por rodada)'
        elif estado['chamadas'] + len(chamadas) > MAX_CHAMADAS_TOTAL:
            recusa = f'limite de {MAX_CHAMADAS_TOTAL} execuções por pedido atingido'

        for c in chamadas:
            nome, args = c.function.name, dict(c.function.arguments or {})
            if recusa:
                resultado = falha('limite', recusa)
            elif nome not in permitidas:
                resultado = falha('nao_autorizada', f'{nome} não está autorizada nesta aplicação')
            else:
                estado['chamadas'] += 1
                try:
                    bruto = await asyncio.wait_for(sessao.call_tool(nome, args), TIMEOUT_FERRAMENTA)
                    resultado = ler_resultado(bruto)
                except asyncio.TimeoutError:
                    resultado = falha('timeout', f'{nome} não respondeu em {TIMEOUT_FERRAMENTA}s')
                estado['evidencias'].append({'ferramenta': nome, 'argumentos': args, 'resultado': resultado})
            if verbose:
                print(f"  [rodada {rodada}] {'ok ' if resultado['ok'] else 'ERRO'} {nome}({args}) -> "
                      f"{resultado.get('dados', resultado.get('erro'))}")
            historico.append({'role': 'tool', 'tool_name': nome,
                              'content': json.dumps(resultado, ensure_ascii=False)})

    return fim('limite_rodadas', motivo=f'resposta final pendente após {MAX_RODADAS} rodadas')


def novo_historico(pedido):
    return [{'role': 'system', 'content': SYSTEM_EXECUTOR}, {'role': 'user', 'content': pedido}]


def mostrar(r):
    print(f"\nstatus: {r['status']} | rodadas: {r['rodadas']} | execuções: {r['chamadas']}")
    if r['motivo']:
        print('motivo:', r['motivo'])
    if r['resposta']:
        print('Resposta:', r['resposta'])

In [ ]:
async def atender(pedido, permitidas=None):
    async with conectar() as sessao:
        historico = novo_historico(pedido)
        r = await rodar_executor(sessao, historico, permitidas)
    mostrar(r)
    return r


r = await atender('Quanto custam 2 camisetas para o Centro?')

**Permissões:** a aplicação decide o que o modelo pode usar. Tire `calcular_total` das permitidas e veja o que acontece
(o modelo nem recebe a ferramenta; se pedir mesmo assim, a chamada é recusada).

In [ ]:
r = await atender('Quanto custam 2 camisetas para o Centro?',
                  permitidas={'consultar_produto', 'consultar_frete'})

**Erro atravessando a conexão:** com quantidade negativa, a função recusa a entrada e o executor deve pedir uma quantidade válida.

In [ ]:
r = await atender('Quanto custam -3 canecas para a Alvorada?')

## Desafio 4 — O servidor ganhou uma função

Acrescentamos `consultar_prazo` **ao servidor** e descobrimos as ferramentas de novo.

> Qual parte do agente deveria precisar de uma alteração?

Repare no que **não** muda: `mcp_para_ollama`, `rodar_executor`, `conectar`.
O que muda é só uma **decisão da aplicação**: autorizar (ou não) a nova ferramenta.

In [ ]:
NOVA_FERRAMENTA = """
PRAZOS = {'centro': 1, 'alvorada': 3}


@mcp.tool()
def consultar_prazo(bairro: str) -> dict[str, Any]:
    '''Consulta o prazo de entrega, em dias úteis, para um bairro.'''
    nome = bairro.strip().lower()
    if nome not in PRAZOS:
        return {'encontrado': False, 'bairro': nome, 'atendidos': sorted(PRAZOS)}
    return {'encontrado': True, 'bairro': nome, 'prazo_dias_uteis': PRAZOS[nome]}
"""

INICIO = "\nif __name__ == '__main__':"
SERVIDOR_V2 = SERVIDOR_V1.replace(INICIO, NOVA_FERRAMENTA + '\n' + INICIO)
arquivo_servidor.write_text(SERVIDOR_V2, encoding='utf-8')

async with conectar() as sessao:
    print('descobertas:', [f.name for f in (await sessao.list_tools()).tools])

In [ ]:
pedido = 'Quanto custam 2 camisetas para a Alvorada e em quantos dias chegam?'

print('=== sem autorizar consultar_prazo ===')
await atender(pedido)

print('\n=== autorizando consultar_prazo ===')
PERMITIDAS = PERMITIDAS | {'consultar_prazo'}
await atender(pedido)

**Para pensar:** sem autorização, o executor inventou um prazo ou disse que não sabe?

## Desafio 5 — Dividir em papéis: executor e revisor (sequência)

| Papel | Recebe | Produz |
|---|---|---|
| **Executor** | pedido, ferramentas autorizadas e seus resultados | resposta + **evidências** |
| **Revisor** | pedido, resposta e evidências (sem ferramentas, histórico próprio) | parecer: total, dados ausentes, afirmações sem suporte |

O revisor pode apontar erro — mas **também pode errar**. Por isso o código confere o total por conta própria.

In [ ]:
SYSTEM_REVISOR = (
    'Você é um revisor de orçamentos. Você NÃO conversa com o cliente. '
    'Confira a RESPOSTA usando só o PEDIDO e as EVIDÊNCIAS (resultados de ferramentas): '
    '1) o total citado é igual ao resultado de calcular_total; '
    '2) dados ausentes (encontrado=false, ok=false) foram informados ao cliente; '
    '3) não há afirmações sem suporte nas evidências (prazos, descontos, estoque, valores). '
    'Responda APENAS um JSON: {"aprovado": true|false, "problemas": ["..."]}'
)


async def revisar(pedido, resposta, evidencias):
    mensagens = [
        {'role': 'system', 'content': SYSTEM_REVISOR},
        {'role': 'user', 'content': json.dumps(
            {'pedido': pedido, 'resposta': resposta, 'evidencias': evidencias}, ensure_ascii=False)},
    ]
    r = await chamar_modelo(model=MODELO_REVISOR, messages=mensagens, format='json',
                            options={'num_predict': 1024})
    try:
        parecer = json.loads(r.message.content)
    except json.JSONDecodeError:
        parecer = {'aprovado': False, 'problemas': ['revisor devolveu JSON inválido'],
                   'bruto': r.message.content}
    return parecer


def conferir_total(resposta, evidencias):
    totais = [e['resultado']['dados']['result'] for e in evidencias
              if e['ferramenta'] == 'calcular_total' and e['resultado']['ok']]
    texto = resposta.replace('.', ',')
    return {t: (f'{t:.0f}' in resposta) or (f'{t:.2f}'.replace('.', ',') in texto) for t in totais}


async def executor_e_revisor(pedido):
    async with conectar() as sessao:
        r = await rodar_executor(sessao, novo_historico(pedido))
    mostrar(r)
    if r['status'] != 'ok':
        return r, None
    parecer = await revisar(pedido, r['resposta'], r['evidencias'])
    print('\nREVISOR:', parecer)
    print('CÓDIGO (total citado na resposta?):', conferir_total(r['resposta'], r['evidencias']))
    return r, parecer


r, parecer = await executor_e_revisor('Quanto custam 2 camisetas para o Centro?')

O revisor percebe um erro? Vamos **forjar** uma resposta errada com as mesmas evidências:

In [ ]:
evidencias = r['evidencias']
for resposta_falsa in ('O total é R$70,00, com entrega amanhã.',
                       'O total é R$78,00.'):
    parecer = await revisar('Quanto custam 2 camisetas para o Centro?', resposta_falsa, evidencias)
    print(f'{resposta_falsa!r}\n  revisor: {parecer}\n  código:  {conferir_total(resposta_falsa, evidencias)}\n')

**Para pensar:** o revisor pegou o total errado **e** o prazo inventado ("entrega amanhã")? Mais papéis significam mais chamadas, tempo e pontos de falha — valeu a pena aqui?

## Desafio 6 — Papéis em paralelo

Pedidos **independentes** podem ser atendidos ao mesmo tempo, cada um com seu histórico, na mesma conexão MCP.
Compare o tempo em sequência e em paralelo.

In [ ]:
PEDIDOS = ['Quanto custam 2 camisetas para o Centro?',
           'Quanto custam 3 canecas para a Alvorada?']


async def sequencia_e_paralelo():
    async with conectar() as sessao:
        t = time.perf_counter()
        for p in PEDIDOS:
            await rodar_executor(sessao, novo_historico(p), verbose=False)
        t_seq = time.perf_counter() - t

        t = time.perf_counter()
        resultados = await asyncio.gather(
            *(rodar_executor(sessao, novo_historico(p), verbose=False) for p in PEDIDOS))
        t_par = time.perf_counter() - t

    for p, r in zip(PEDIDOS, resultados):
        print(f'{p}\n  -> {r["status"]}: {r["resposta"]}')
    print(f'\nsequência: {t_seq:.1f}s | paralelo: {t_par:.1f}s')


await sequencia_e_paralelo()

Referências: 2 camisetas no Centro = **R$78**; 3 canecas na Alvorada = 3 × 22 + 12 = **R$78**.

Três formas de organizar o trabalho:
- **Sequência** — um papel termina e passa o resultado ao próximo (Desafio 5).
- **Paralelo** — papéis independentes ao mesmo tempo (Desafio 6).
- **Coordenação** — um supervisor distribui tarefas e reúne resultados (o coordenador + especialista da Aula 8).

## Desafio 7 — Sua versão

Use o tema da sua aplicação da Aula 7 (loja fictícia, itens de um jogo, materiais de um evento…):
1. escreva um servidor MCP com **três ferramentas** (duas consultas + um cálculo) e validação dentro das funções;
2. descubra as ferramentas e autorize só as que fazem sentido;
3. mostre **um pedido completo**, **um dado ausente** e **um erro de entrada** atravessando a conexão;
4. acrescente uma quarta ferramenta ao servidor e mostre que o laço não mudou;
5. passe a resposta pelo revisor e compare com a conferência do código.

Dica: copie `SERVIDOR_V1`, troque os dicionários e as docstrings, e grave em outro arquivo (ex.: `servidor_mcp/jogo.py`),
ajustando `parametros`.

---
**O que conseguimos separar?**
- O **servidor** publica e executa as ferramentas.
- A **aplicação** descobre, autoriza e coordena o uso.

**Próxima aula:** avaliar resultados, segurança e ética.